# R2 Bucket Test

Verify that boto3 can authenticate and reach the `cytemaps` R2 bucket.

In [1]:
import os
from pathlib import Path

import boto3
from dotenv import load_dotenv

load_dotenv(dotenv_path=Path("..") / ".env")

BUCKET = os.environ["BUCKET"]
ENDPOINT_URL = os.environ["ENDPOINT_URL"]

# Test file name
KEY = "test.txt"

client = boto3.client(
    "s3",
    endpoint_url=ENDPOINT_URL,
    aws_access_key_id=os.environ["AWS_ACCESS_KEY_ID"],
    aws_secret_access_key=os.environ["AWS_SECRET_ACCESS_KEY"],
)

print(f"Endpoint : {ENDPOINT_URL}")
print(f"Bucket   : {BUCKET}")

Endpoint : https://8a09aa0b1ce3614588bfcd221d8c183e.eu.r2.cloudflarestorage.com
Bucket   : cytemaps


## 1. List all objects

In [2]:
paginator = client.get_paginator("list_objects_v2")
objects = []
for page in paginator.paginate(Bucket=BUCKET):
    objects.extend(page.get("Contents") or [])

if not objects:
    print("Bucket is empty.")
else:
    print(f"{len(objects)} object(s):")
    for obj in objects:
        size_mb = obj["Size"] / 1024 / 1024
        uploaded = obj["LastModified"].strftime("%Y-%m-%d %H:%M")
        print(f"  {obj['Key']}  ({size_mb:.1f} MB)  {uploaded}")

3 object(s):
  cytetype/SRX12708356_cytetype_annotated.h5ad  (14.5 MB)  2026-05-05 13:45
  cytetype/SRX22996378_cytetype_annotated.h5ad  (253.5 MB)  2026-05-05 12:10
  test.txt  (0.0 MB)  2026-05-04 14:17


## 2. Filter by prefix

In [17]:
PREFIX = "cytetype/"

paginator = client.get_paginator("list_objects_v2")
filtered = []
for page in paginator.paginate(Bucket=BUCKET, Prefix=PREFIX):
    filtered.extend(page.get("Contents") or [])

if not filtered:
    print(f"No objects found under '{PREFIX}'.")
else:
    print(f"{len(filtered)} object(s) under '{PREFIX}':")
    for obj in filtered:
        size_mb = obj["Size"] / 1024 / 1024
        print(f"  {obj['Key']}  ({size_mb:.1f} MB)")

No objects found under 'cytetype/'.


## 3. Upload a test file

In [18]:
# import io

# body = b"hello from r2_test.ipynb"

# client.put_object(Bucket=BUCKET, Key=KEY, Body=body)
# print(f"Uploaded '{KEY}' ({len(body)} bytes) to r2://{BUCKET}/{KEY}")

SyntaxError: unterminated string literal (detected at line 2) (1275901648.py, line 2)

## 4. Read the test file back

In [ ]:
response = client.get_object(Bucket=BUCKET, Key=KEY)
content = response["Body"].read().decode()
print(f"Contents of '{KEY}': {content!r}")

Contents of 'test.txt': 'hello from r2_test.ipynb'


## 5. Download the test file to repo root

In [ ]:
# from shared.repo import REPO_ROOT

# dest = REPO_ROOT / "tmp" / KEY
# client.download_file(BUCKET, KEY, str(dest))
# print(f"Downloaded '{KEY}' -> {dest}")
# print(f"Contents: {dest.read_text()!r}")

Downloaded 'test.txt' -> /Users/otodreas/Desktop/Work/Nygen/scBaseCount_Pipeline/tmp/test.txt
Contents: 'hello from r2_test.ipynb'
